In [32]:
import lzma
import pickle
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

In [33]:
with lzma.open("../../data/cleaned/starling_cleaned.pkl.xz", "rb") as f:
    starling = pickle.load(f)

with lzma.open("../../data/cleaned/unihan_cleaned.pkl.xz", "rb") as f:
    unihan = pickle.load(f)

- Filter by radicals
- Top 3000 most common characters

In [34]:
import opencc

# Valid pinyin: a-z plus toned a/e/i/o/u/ü
_VALID_PINYIN_CHARS = re.compile(
    r'[^a-zāáǎàēéěèīíǐìōóǒòūúǔùüǖǘǚǜ]'
)

def take_non_missing_value(row):
    return "; ".join({unicodedata.normalize("NFC", str(value)) for value in row if pd.notna(value)})

def split_pinyin(s):
    parts = re.split(r'[;\s]+', s.strip())
    seen = set()
    result = []
    for p in parts:
        p = p.strip()
        if p and p not in seen:
            seen.add(p)
            result.append(p)
    return result if result else [s]

def clean_pinyin(s):
    """Remove invalid characters; return None if nothing valid remains."""
    cleaned = _VALID_PINYIN_CHARS.sub('', s)
    return cleaned if cleaned else None

_converter = opencc.OpenCC("t2s")

# Build radical map from kRSUnicode in Unihan_IRGSources.txt
# Format: radical_number.strokes  (prime suffix = simplified-form radical, still same number)
_radical_map: dict[str, str] = {}
with open("../../data/raw/Unihan_IRGSources.txt") as _f:
    for _line in _f:
        if _line.startswith('#') or not _line.strip():
            continue
        _parts = _line.strip().split('\t')
        if len(_parts) == 3 and _parts[1] == 'kRSUnicode':
            _m = re.match(r"(\d+)'?\.", _parts[2].split()[0])
            if _m:
                _char = chr(int(_parts[0][2:], 16))
                _radical_char = chr(0x2F00 + int(_m.group(1)) - 1)
                _radical_map[_char] = _radical_char

merged = unihan.merge(starling, on="Character", how="outer", suffixes=("_unihan", "_starling"))

merged["definition"] = merged[["kDefinition", "English meaning"]].apply(take_non_missing_value, axis=1)
merged["pinyin"]     = merged[["kMandarin", "Modern (Beijing) reading"]].apply(take_non_missing_value, axis=1)

merged = merged.rename(columns={
    "Character":   "hanzi",
    "kHangul":     "hangul",
    "kKorean":     "anglo_hangul",
    "kJapanese":   "katakana",
    "kJapaneseOn": "anglo_katakana",
})[["hanzi", "definition", "pinyin", "hangul", "anglo_hangul", "katakana", "anglo_katakana"]]

merged["simplified"] = merged["hanzi"].apply(lambda x: _converter.convert(x) if pd.notna(x) else x)
merged["radical"]    = merged["hanzi"].map(_radical_map)

# Split pinyin on ";" and whitespace, deduplicate, explode to one reading per row
merged["pinyin"] = merged["pinyin"].apply(split_pinyin)
merged = merged.explode("pinyin").reset_index(drop=True)

# Strip invalid characters, drop rows with nothing left
merged["pinyin"] = merged["pinyin"].apply(clean_pinyin)
merged = merged.dropna(subset=["pinyin"]).reset_index(drop=True)

# Recompute derived columns from single pinyin per row
merged["decomposed_pinyin"] = merged["pinyin"].apply(
    lambda x: unicodedata.normalize("NFD", x)
)
merged["toneless_pinyin"] = merged["decomposed_pinyin"].apply(
    lambda x: "".join(re.findall(r'[a-z]+', x))
)

merged = merged[["hanzi", "simplified", "radical", "definition", "pinyin", "decomposed_pinyin", "toneless_pinyin", "hangul", "anglo_hangul", "katakana", "anglo_katakana"]]

merged


,hanzi,simplified,radical,definition,pinyin,decomposed_pinyin,toneless_pinyin,hangul,anglo_hangul,katakana,anglo_katakana
0,Ф,Ф,NaN,"to send, cause (?)",bēng,bēng,beng,NaN,NaN,NaN,NaN
1,Щ,Щ,NaN,"be robust, strong",bì,bì,bi,NaN,NaN,NaN,NaN
2,к,к,NaN,"flask, bottle (for wine)",yǒu,yǒu,you,NaN,NaN,NaN,NaN
3,へ,へ,NaN,"be grieved, sad",daō,daō,dao,NaN,NaN,NaN,NaN
4,一,一,⼀,"be one, single, whole; one; a, an; alone",yī,yī,yi,일,IL,イチ,ICHI
...,...,...,...,...,...,...,...,...,...,...,...
8560,龜,龟,⿔,"turtle, tortoise; bone oracle in general; turt...",guī,guī,gui,구,KWU,キ,KI
8561,龝,龝,⿔,"autumn, fall; year",qiū,qiū,qiu,추,CHWU,シュウ,SHUU
8562,龠,龠,⿕,"flute; pipe, ancient measure; Kangxi radical 214",yuè,yuè,yue,약,YAK,ヤク,YAKU
8563,龢,龢,⿕,"in harmony; calm, peaceful",hé,hé,he,화,HWA,カ,KA


In [ ]:
import json
from pathlib import Path

import ollama

ANNO_MODEL = "qwen2.5:7b"
ANNO_CACHE = Path("../../data/cleaned/annotations.json")
ANNO_PROMPT = """\
You are an expert in Chinese language and culture.
Given a Chinese character, reply with a single JSON object — no markdown, no extra text — with exactly these keys:
  "meaning":   core English meaning (<=8 words)
  "name_use":  suitability as a given-name character: "good", "neutral", or "avoid"
  "note":      important cultural/negative connotation, or "" if none (<=10 words)

Character: {char}
Pinyin: {pinyin}
Dictionary definition: {definition}
"""

# Load existing cache so we can resume interrupted runs
_anno_cache: dict[str, dict] = {}
if ANNO_CACHE.exists():
    _anno_cache = json.loads(ANNO_CACHE.read_text())

def _annotate(char: str, pinyin: str, definition: str) -> dict:
    key = f"{char}|{pinyin}"
    if key in _anno_cache:
        return _anno_cache[key]
    prompt = ANNO_PROMPT.format(char=char, pinyin=pinyin, definition=definition)
    resp = ollama.chat(
        model=ANNO_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    text = resp.message.content.strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        result = json.loads(m.group()) if m else {"meaning": "", "name_use": "", "note": text[:80]}
    _anno_cache[key] = result
    return result

# Annotate unique (hanzi, pinyin) pairs — safe to re-run, resumes from cache
unique_chars = merged[["hanzi", "pinyin", "definition"]].drop_duplicates(subset=["hanzi", "pinyin"])
total = len(unique_chars)
for i, (_, row) in enumerate(unique_chars.iterrows()):
    key = f"{row['hanzi']}|{row['pinyin']}"
    if key not in _anno_cache:
        _annotate(row["hanzi"], row["pinyin"], row["definition"])
        ANNO_CACHE.write_text(json.dumps(_anno_cache, ensure_ascii=False, indent=2))
    if (i + 1) % 100 == 0 or (i + 1) == total:
        print(f"  {i+1}/{total} annotated")

print("Done.")


In [4]:
merged.to_pickle("../../data/cleaned/merged.pkl.xz")